# Lesson 3: Wrap a PyTorch model

**You will learn:** how to turn a PyTorch network into a subject with `BrainScoreModel`, and how to record its activity as a stand-in for a brain region.
**Requires:** CPU only, no downloads, under a minute.

Writing `participate` by hand ([Lesson 1](01_your_first_subject.ipynb)) works for any system. For neural networks, Brain-Score provides `BrainScoreModel`, a ready-made subject assembled from parts:

| Part | Job |
| --- | --- |
| `extractor` | Prepares inputs, runs the network, and returns the activity of requested layers. One per input type (`vision`, `text`, `audio`). |
| `region_layer_map` | Says which layer stands in for which brain region, for example `{'IT': 'layer4'}`. |
| `answer`, `policy`, `intervention` | Optional parts for behavior, closed-loop control and perturbations (later lessons). |

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


In [3]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

## Layer paths

A layer is named by its PyTorch module path. List the paths of your network with `named_modules()`.

In [4]:
[name for name, _ in net.named_modules() if name]

['0', '1', '2', '3', '4']

## Build the extractor

`VisionWrapper` is the extractor for images. Give it the network and a preprocessing function. Text and audio have their own wrappers (`TextWrapper`, `AudioWrapper`) with the same calling convention.

In [5]:
from brainscore.model_helpers.vision_wrapper import VisionWrapper

extractor = VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)

## Build the subject

The keys of `extractor` declare the input channels. The keys of `region_layer_map` declare the output channels, as `neural:<region>`. The mapping below is an arbitrary choice for illustration; in practice a benchmark tests whether a mapping is a good one.

In [6]:
from brainscore_core import BrainScoreModel

model = BrainScoreModel(
    identifier='toy-cnn',
    model=net,
    extractor={'vision': extractor},
    region_layer_map={'V1': '0', 'IT': '4'},
)
print('inputs: ', model.in_channels)
print('outputs:', model.out_channels)

inputs:  {'vision'}
outputs: {'neural:V1', 'neural:IT'}


## Record a region through a session

`BufferedSession` can be built directly from a stimulus table. Ask for the `neural:IT` channel, let the model participate, and collect the result. This is exactly what a neural benchmark does.

In [7]:
from brainscore_core.streaming import BufferedSession

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'])
model.participate(session)
activity = session.collect()
model.reset()  # clear recording state before the next, independent run
activity

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

<xarray.NeuroidAssembly (presentation: 8, neuroid: 8)>
array([[ 0.08637821,  0.1945948 , -0.07783568, -0.00804976,  0.13398749,
        -0.25402907,  0.16535991, -0.18919885],
       [ 0.05250589,  0.21703452, -0.10035843,  0.0045949 ,  0.14602682,
        -0.24028078,  0.19894187, -0.21468118],
       [ 0.09005468,  0.19688796, -0.08476483, -0.01423864,  0.13407512,
        -0.25592554,  0.15970035, -0.1967448 ],
       [ 0.05471925,  0.2159542 , -0.09632318,  0.00447319,  0.1416466 ,
        -0.2433597 ,  0.19549373, -0.21259372],
       [ 0.08946367,  0.1956514 , -0.08128732, -0.01328416,  0.13585837,
        -0.2522271 ,  0.15957925, -0.19401188],
       [ 0.05612481,  0.21786205, -0.09854188,  0.00177641,  0.14088342,
        -0.24972416,  0.19509071, -0.21289885],
       [ 0.08988599,  0.19478275, -0.08350355, -0.01152686,  0.13315797,
        -0.25586405,  0.16325364, -0.19533177],
       [ 0.0549632 ,  0.2154096 , -0.10030724,  0.00674276,  0.14031747,
        -0.24548921,  0.19392696, -0.21395905]], dtype=float32)
Coordinates:
  * neuroid                       (neuroid) MultiIndex
  - neuroid_num                   (neuroid) int64 0 1 2 3 4 5 6 7
  - model                         (neuroid) object 'toy-cnn' ... 'toy-cnn'
  - layer                         (neuroid) object '4' '4' '4' ... '4' '4' '4'
  - channel                       (neuroid) int64 0 1 2 3 4 5 6 7
  - channel_x                     (neuroid) float64 nan nan nan ... nan nan nan
  - channel_y                     (neuroid) float64 nan nan nan ... nan nan nan
  - neuroid_id                    (neuroid) object 'toy-cnn.4.0' ... 'toy-cnn...
  * presentation                  (presentation) MultiIndex
  - microsaccade_shift_y_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - microsaccade_shift_x_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - stimulus_id                   (presentation) object 'image_0' ... 'image_7'
  - microsaccade_shift_x_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - label                         (presentation) object 'bright' ... 'dark'
  - image_path                    (presentation) object '/var/folders/h6/g53m...
  - microsaccade_shift_y_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0

The result is a `NeuroidAssembly`: a labeled array with one row per stimulus (`presentation`) and one column per recorded unit (`neuroid`). Its coordinates keep the stimulus IDs and layer names.

## `participate` and `process`

`participate(session)` is the method every benchmark and experiment calls. `BrainScoreModel` also offers `process(...)`, a shortcut for working interactively in a notebook. It does the same computation without a session.

In [8]:
model.start_recording('IT')
same_activity = model.process(stimuli)
model.reset()
np.allclose(same_activity.values, activity.values)

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

True

## Summary

- `BrainScoreModel` builds a subject from an **extractor** (per input type) and a **region_layer_map**.
- Channels follow from those parts: `extractor={'vision': ...}` gives the `vision` input; `region_layer_map={'IT': ...}` gives the `neural:IT` output.
- Benchmarks call `participate(session)`. Use `process()` only as an interactive shortcut.
- Call `reset()` between independent runs.

**Next:** [Lesson 4](04_bring_an_existing_model.ipynb) shows how existing Brain-Score models become subjects without rewriting them.